# 🧪 A/B Тест: Бонусний лист після реєстрації → Перший депозит

**Гіпотеза:** Юзери, що отримали лист із бонусами через 12 год після реєстрації (група A), матимуть вищу конверсію в перший депозит ніж контрольна група (B).

---
### Структура даних
Два масиви з сумами депозитів юзерів:
- `0` = юзер **не зробив** депозит
- `число > 0` = сума першого депозиту в €

## 0. Встановлення бібліотек
Запусти один раз, потім можна закоментувати.

In [ ]:
# !pip install numpy pandas scipy matplotlib statsmodels

## 1. Імпорти

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import levene, mannwhitneyu, shapiro
import statsmodels.stats.proportion as smprop
import statsmodels.stats.power as smpow
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

print('✅ Бібліотеки завантажено')

## 2. Завантаження даних

**Розкоментуй один варіант під свої дані:**

| Варіант | Опис |
|---------|------|
| А | Один CSV, дві колонки `group_a` / `group_b` |
| Б | Один CSV, колонки `group` і `deposit` |
| В | Два окремі CSV-файли |
| Г | Excel-файл |
| Д | Два Python-списки вручну |


In [ ]:
# ── Варіант А: один CSV, дві колонки ──────────────────────────
# df = pd.read_csv('deposits.csv')
# group_a_raw = df['group_a'].dropna().values
# group_b_raw = df['group_b'].dropna().values

# ── Варіант Б: один CSV, колонки group + deposit ───────────────
# df = pd.read_csv('all_users.csv')
# group_a_raw = df[df['group'] == 'A']['deposit'].fillna(0).values
# group_b_raw = df[df['group'] == 'B']['deposit'].fillna(0).values

# ── Варіант В: два окремі CSV ──────────────────────────────────
# group_a_raw = pd.read_csv('group_a.csv')['deposit'].fillna(0).values
# group_b_raw = pd.read_csv('group_b.csv')['deposit'].fillna(0).values

# ── Варіант Г: Excel ───────────────────────────────────────────
# df = pd.read_excel('deposits.xlsx')
# group_a_raw = df[df['group'] == 'A']['deposit'].fillna(0).values
# group_b_raw = df[df['group'] == 'B']['deposit'].fillna(0).values

# ── Варіант Д: вручну ──────────────────────────────────────────
# group_a_raw = [15, 0, 20, 0, 10, ...]   # 0 = без депозиту
# group_b_raw = [0, 12, 0, 25, 0, ...]


# ════════════════════════════════════════════════════════════════
#  ДЕМО: синтетичні дані під твої агрегати
#  Видали цей блок після підстановки своїх даних!
# ════════════════════════════════════════════════════════════════
np.random.seed(42)
n_a_total, n_dep_a_demo, sum_a_demo = 17955, 344, 5400.05
n_b_total, n_dep_b_demo, sum_b_demo = 17907, 302, 4652.38
mean_a_demo = sum_a_demo / n_dep_a_demo
mean_b_demo = sum_b_demo / n_dep_b_demo

dep_vals_a = np.random.lognormal(np.log(mean_a_demo) - 0.5, 1.0, n_dep_a_demo)
dep_vals_a = dep_vals_a / dep_vals_a.mean() * mean_a_demo
dep_vals_b = np.random.lognormal(np.log(mean_b_demo) - 0.5, 1.0, n_dep_b_demo)
dep_vals_b = dep_vals_b / dep_vals_b.mean() * mean_b_demo

group_a_raw = np.concatenate([dep_vals_a, np.zeros(n_a_total - n_dep_a_demo)])
group_b_raw = np.concatenate([dep_vals_b, np.zeros(n_b_total - n_dep_b_demo)])
np.random.shuffle(group_a_raw)
np.random.shuffle(group_b_raw)
# ════════════════════════════════════════════════════════════════

print(f'✅ Дані завантажено: A={len(group_a_raw):,} | B={len(group_b_raw):,}')

## 3. Підготовка даних

In [ ]:
a_all = np.array(group_a_raw, dtype=float)
b_all = np.array(group_b_raw, dtype=float)

# Тільки юзери, що зробили депозит
a_dep = a_all[a_all > 0]
b_dep = b_all[b_all > 0]

n_a, n_b         = len(a_all), len(b_all)
ndep_a, ndep_b   = len(a_dep), len(b_dep)
conv_a = ndep_a / n_a
conv_b = ndep_b / n_b

print(f'Група A: {n_a:,} юзерів, {ndep_a:,} депозитів, конверсія {conv_a:.3%}')
print(f'Група B: {n_b:,} юзерів, {ndep_b:,} депозитів, конверсія {conv_b:.3%}')

## 4. Метрики

In [ ]:
def compute_metrics(dep_arr, all_arr, label):
    return {
        'Група':       label,
        'К-ть юзерів': len(all_arr),
        'К-ть депозитів': len(dep_arr),
        'Конверсія':   f"{len(dep_arr)/len(all_arr):.3%}",
        'Середній деп': f"€{dep_arr.mean():.2f}",
        'Медіана деп':  f"€{np.median(dep_arr):.2f}",
        'Стд відх':     f"€{dep_arr.std(ddof=1):.2f}",
        'P10':          f"€{np.percentile(dep_arr, 10):.2f}",
        'P90':          f"€{np.percentile(dep_arr, 90):.2f}",
        'Сума деп':     f"€{dep_arr.sum():.2f}",
        'Skewness':     f"{stats.skew(dep_arr):.3f}",
    }

ma = compute_metrics(a_dep, a_all, 'A (тест)')
mb = compute_metrics(b_dep, b_all, 'B (контроль)')

df_metrics = pd.DataFrame([ma, mb]).set_index('Група').T
display(df_metrics)

## 5. Перевірка нормальності (Shapiro-Wilk)

**Навіщо:** T-тест і Z-тест для середніх працюють коректно при нормальному розподілі.  
Суми депозитів зазвичай **не** нормальні (є викиди, правий хвіст).  
Якщо `p < 0.05` → дані не нормальні → Mann-Whitney надійніший.


In [ ]:
# Shapiro точний тільки при n < 5000, беремо вибірку
sample_a = a_dep[:5000] if len(a_dep) > 5000 else a_dep
sample_b = b_dep[:5000] if len(b_dep) > 5000 else b_dep

sw_a = shapiro(sample_a)
sw_b = shapiro(sample_b)

normal_a = sw_a.pvalue > 0.05
normal_b = sw_b.pvalue > 0.05
both_normal = normal_a and normal_b

print(f"Група A: W={sw_a.statistic:.4f}, p={sw_a.pvalue:.4f} → {'✅ нормальний' if normal_a else '❌ НЕ нормальний'}")
print(f"Група B: W={sw_b.statistic:.4f}, p={sw_b.pvalue:.4f} → {'✅ нормальний' if normal_b else '❌ НЕ нормальний'}")
print()
print(f"→ Рекомендований тест для середнього: "
      f"{'t-тест (дані нормальні)' if both_normal else 'Mann-Whitney (дані не нормальні)'}")

## 6. Рівність дисперсій (Levene)

**Навіщо:** Визначає, який t-тест використовувати:
- `p > 0.05` → дисперсії рівні → можна **Стьюдента**
- `p ≤ 0.05` → дисперсії різні → треба **Велча**

> На практиці **Велч завжди безпечніший** — він коректний в обох випадках.

In [ ]:
lev_stat, p_levene = levene(a_dep, b_dep)
equal_var = p_levene > 0.05
chosen_test = 'Стьюдента' if equal_var else 'Велча'

print(f"Levene: F={lev_stat:.4f}, p={p_levene:.4f}")
print(f"→ Дисперсії {'РІВНІ ✅ → можна Стьюдента' if equal_var else 'РІЗНІ ❌ → потрібен Велч'}")
print(f"→ Авто-вибір: t-тест {chosen_test}")

## 7. Z-тест для конверсії (statsmodels)

**Коли:** метрика — це частка (конверсія, CTR, churn).  
**Не підходить** для середніх сум — там потрібен t-тест.

```
H₀: p_A = p_B   (різниці немає)
H₁: p_A ≠ p_B   (двостороннє)
H₁: p_A > p_B   (одностороннє)
```

In [ ]:
# statsmodels робить все одним викликом
z_stat, p_z_two = smprop.proportions_ztest(
    count=[ndep_a, ndep_b],
    nobs=[n_a, n_b],
    alternative='two-sided'
)
_, p_z_one = smprop.proportions_ztest(
    count=[ndep_a, ndep_b],
    nobs=[n_a, n_b],
    alternative='larger'    # H1: A > B
)

# 95% CI для різниці пропорцій (метод Newcomb — точніший за Wald)
ci_low, ci_high = smprop.confint_proportions_2indep(
    count1=ndep_a, nobs1=n_a,
    count2=ndep_b, nobs2=n_b,
    method='newcomb'
)

lift_conv = (conv_a - conv_b) / conv_b * 100

print(f"Z-статистика:          {z_stat:.4f}")
print(f"p-value (2-side):      {p_z_two:.4f}  {'✅ Значущий' if p_z_two < 0.05 else '⚠️  На межі' if p_z_two < 0.1 else '❌ НЕ значущий'} (α=0.05)")
print(f"p-value (1-side A>B):  {p_z_one:.4f}  {'✅ Значущий' if p_z_one < 0.05 else '❌ НЕ значущий'}")
print(f"95% CI різниці:        [{ci_low:+.5f}, {ci_high:+.5f}]")
print(f"  → CI {'не містить 0 — різниця значуща ✅' if ci_low > 0 else 'містить 0 — різниця НЕ значуща ❌'}")
print(f"Ліфт конверсії:        {lift_conv:+.2f}%")

## 8. T-тест для середнього депозиту

### Велч vs Стьюдент — у чому різниця?

| | Стьюдент | Велч |
|---|---|---|
| Припущення | σ_A = σ_B | σ_A може ≠ σ_B |
| Ступені свободи | n_A + n_B - 2 | формула Satterthwaite (менше) |
| Коли використовувати | Levene p > 0.05 | **завжди безпечний** |
| `equal_var=` | `True` | `False` |

> Велч = консервативніший (більший CI, важче отримати значущість), але чесніший.

In [ ]:
# Велч (рекомендований)
t_welch, p_welch = stats.ttest_ind(a_dep, b_dep, equal_var=False)

# Стьюдент
t_student, p_student = stats.ttest_ind(a_dep, b_dep, equal_var=True)

# Авто-вибір за тестом Левена
t_auto, p_auto = (t_student, p_student) if equal_var else (t_welch, p_welch)

# 95% CI для різниці середніх (Велч)
sa2, sb2 = a_dep.std(ddof=1)**2, b_dep.std(ddof=1)**2
se_diff  = np.sqrt(sa2/ndep_a + sb2/ndep_b)
df_w     = (sa2/ndep_a + sb2/ndep_b)**2 / \
           ((sa2/ndep_a)**2/(ndep_a-1) + (sb2/ndep_b)**2/(ndep_b-1))
t_crit   = stats.t.ppf(0.975, df=df_w)
diff_mean = a_dep.mean() - b_dep.mean()
ci_mean  = (diff_mean - t_crit*se_diff, diff_mean + t_crit*se_diff)

lift_mean = diff_mean / b_dep.mean() * 100

print("═" * 50)
print("  T-ТЕСТ ВЕЛЧА (equal_var=False)")
print("═" * 50)
print(f"  t = {t_welch:.4f},  df = {df_w:.1f}")
print(f"  p = {p_welch:.4f}  {'✅ Значущий' if p_welch < 0.05 else '❌ НЕ значущий'}")
print(f"  95% CI: [€{ci_mean[0]:+.3f}, €{ci_mean[1]:+.3f}]")
print()
print("═" * 50)
print("  T-ТЕСТ СТЬЮДЕНТА (equal_var=True)")
print("═" * 50)
print(f"  t = {t_student:.4f},  df = {ndep_a + ndep_b - 2}")
print(f"  p = {p_student:.4f}  {'✅ Значущий' if p_student < 0.05 else '❌ НЕ значущий'}")
print()
print(f"  Авто-вибір (за Levene): t-тест {chosen_test}")
print(f"  Ліфт середнього деп: {lift_mean:+.2f}%")

## 9. Mann-Whitney U — для медіани (непараметричний)

**Коли:** дані не нормальні, є викиди (типово для депозитів).  
**Що перевіряє:** чи розподіли A і B однакові (по суті — різниця медіан).  
**Effect size r:** < 0.1 малий, 0.1–0.3 середній, > 0.3 великий.

In [ ]:
u_stat, p_mw = mannwhitneyu(a_dep, b_dep, alternative='two-sided')
_, p_mw_one  = mannwhitneyu(a_dep, b_dep, alternative='greater')

z_mw = stats.norm.ppf(1 - p_mw / 2)
r_mw = z_mw / np.sqrt(ndep_a + ndep_b)

print(f"U-статистика:          {u_stat:.0f}")
print(f"p-value (2-side):      {p_mw:.4f}  {'✅ Значущий' if p_mw < 0.05 else '❌ НЕ значущий'}")
print(f"p-value (1-side A>B):  {p_mw_one:.4f}")
print(f"Effect size r:         {r_mw:.4f}  ({'малий' if abs(r_mw) < 0.1 else 'середній' if abs(r_mw) < 0.3 else 'великий'})")
print(f"Медіана A: €{np.median(a_dep):.2f}  |  Медіана B: €{np.median(b_dep):.2f}")

## 10. Sample size — скільки юзерів потрібно

**Рахуємо через statsmodels** — використовує Cohen's h як effect size для пропорцій.

```
Cohen's h = 2×arcsin(√p_test) − 2×arcsin(√p_base)
```

Параметри: **α = 0.05**, **power = 80%** (стандарт для A/B тестів).

In [ ]:
def sample_size(p_base, lift_pct, alpha=0.05, power=0.80):
    """n на групу через statsmodels NormalIndPower (Cohen's h)."""
    p_test = p_base * (1 + lift_pct / 100)
    h = 2*np.arcsin(np.sqrt(p_test)) - 2*np.arcsin(np.sqrt(p_base))
    n = smpow.NormalIndPower().solve_power(
        effect_size=abs(h), alpha=alpha, power=power, alternative='two-sided'
    )
    return int(np.ceil(n))

# Поточна реальна потужність
h_actual = 2*np.arcsin(np.sqrt(conv_a)) - 2*np.arcsin(np.sqrt(conv_b))
actual_power = smpow.NormalIndPower().solve_power(
    effect_size=abs(h_actual), alpha=0.05, nobs1=min(n_a, n_b), alternative='two-sided'
)

n_needed = sample_size(conv_b, lift_conv)
print(f"Поточна потужність тесту: {actual_power:.1%}  "
      f"{'✅ ≥ 80%' if actual_power >= 0.8 else '❌ < 80% — тест недостатньо потужний'}")
print(f"Потрібно для ліфту {lift_conv:.1f}%: {n_needed:,} на групу ({n_needed*2:,} всього)\n")

# Таблиця
mde_values = [5, 10, round(lift_conv, 1), 15, 20, 30]
rows = []
for mde in mde_values:
    n = sample_size(conv_b, mde)
    rows.append({
        'MDE (ліфт %)': f'{mde:+.1f}%',
        'n на групу': f'{n:,}',
        'Всього': f'{n*2:,}',
        'Поточна вибірка достатня?': '✅ Так' if min(n_a, n_b) >= n else '❌ Ні'
    })

display(pd.DataFrame(rows))

## 11. Візуалізація

In [ ]:
COLOR_A, COLOR_B = '#3B8BD4', '#73726c'

fig = plt.figure(figsize=(18, 13))
fig.patch.set_facecolor('#f8f8f6')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

def annotate_sig(ax, p, x=0.5, y=0.97):
    color = 'green' if p < 0.05 else ('orange' if p < 0.1 else 'red')
    label = '✅ p < 0.05' if p < 0.05 else ('⚠️ p ≈ 0.05' if p < 0.1 else '❌ p > 0.05')
    ax.text(x, y, f'p = {p:.3f}  {label}', transform=ax.transAxes,
            ha='center', va='top', fontsize=9, color=color,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85))

# 1. Конверсія
ax1 = fig.add_subplot(gs[0, 0])
se_pool = np.sqrt(conv_a*(1-conv_a)/n_a + conv_b*(1-conv_b)/n_b)
bars = ax1.bar(['A (тест)', 'B (контроль)'], [conv_a*100, conv_b*100],
               color=[COLOR_A, COLOR_B], alpha=0.8, width=0.5,
               yerr=[1.96*se_pool*100]*2, capsize=7,
               error_kw={'linewidth': 1.5})
ax1.set_title('Конверсія рег→деп', fontweight='bold')
ax1.set_ylabel('Конверсія, %')
for bar, val in zip(bars, [conv_a, conv_b]):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.04,
             f'{val:.3%}', ha='center', va='bottom', fontsize=10, fontweight='bold')
annotate_sig(ax1, p_z_two)
ax1.set_facecolor('#ffffff')
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

# 2. Середній депозит
ax2 = fig.add_subplot(gs[0, 1])
se_a = a_dep.std(ddof=1) / np.sqrt(ndep_a)
se_b = b_dep.std(ddof=1) / np.sqrt(ndep_b)
bars2 = ax2.bar(['A (тест)', 'B (контроль)'], [a_dep.mean(), b_dep.mean()],
                color=[COLOR_A, COLOR_B], alpha=0.8, width=0.5,
                yerr=[1.96*se_a, 1.96*se_b], capsize=7,
                error_kw={'linewidth': 1.5})
ax2.set_title(f'Середній деп | t-тест {chosen_test}', fontweight='bold')
ax2.set_ylabel('€')
for bar, val in zip(bars2, [a_dep.mean(), b_dep.mean()]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'€{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
annotate_sig(ax2, p_auto)
ax2.text(0.5, 0.85, f'Levene p={p_levene:.3f} → {chosen_test}',
         transform=ax2.transAxes, ha='center', fontsize=8, color='gray')
ax2.set_facecolor('#ffffff')
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

# 3. Boxplot
ax3 = fig.add_subplot(gs[0, 2])
bp = ax3.boxplot([a_dep, b_dep], labels=['A (тест)', 'B (контроль)'],
                 patch_artist=True,
                 medianprops=dict(color='white', linewidth=2.5),
                 flierprops=dict(marker='.', markersize=2, alpha=0.3),
                 whiskerprops=dict(linewidth=1.2),
                 capprops=dict(linewidth=1.2))
bp['boxes'][0].set_facecolor(COLOR_A); bp['boxes'][0].set_alpha(0.75)
bp['boxes'][1].set_facecolor(COLOR_B); bp['boxes'][1].set_alpha(0.75)
ax3.set_title('Boxplot депозитів | Mann-Whitney', fontweight='bold')
ax3.set_ylabel('Сума депозиту, €')
annotate_sig(ax3, p_mw)
ax3.text(0.5, 0.85, f'effect r={r_mw:.3f}',
         transform=ax3.transAxes, ha='center', fontsize=8, color='gray')
ax3.set_facecolor('#ffffff')
ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)

# 4. Гістограма
ax4 = fig.add_subplot(gs[1, 0:2])
cap = np.percentile(np.concatenate([a_dep, b_dep]), 95)
bins = np.linspace(0, cap, 45)
ax4.hist(a_dep[a_dep <= cap], bins=bins, alpha=0.6, color=COLOR_A,
         label=f'A (n={ndep_a})', density=True)
ax4.hist(b_dep[b_dep <= cap], bins=bins, alpha=0.5, color=COLOR_B,
         label=f'B (n={ndep_b})', density=True)
for val, color, ls, label in [
    (a_dep.mean(),       COLOR_A, '--', f'Сер.A €{a_dep.mean():.2f}'),
    (b_dep.mean(),       COLOR_B, '--', f'Сер.B €{b_dep.mean():.2f}'),
    (np.median(a_dep),   COLOR_A, ':',  f'Мед.A €{np.median(a_dep):.2f}'),
    (np.median(b_dep),   COLOR_B, ':',  f'Мед.B €{np.median(b_dep):.2f}')]:
    ax4.axvline(val, color=color, linestyle=ls, linewidth=1.8, label=label)
skew_a = stats.skew(a_dep)
ax4.set_title(f'Розподіл депозитів (до 95 перцентиля) | Skewness A={skew_a:.2f}'
              f' → {"Mann-Whitney надійніший" if abs(skew_a) > 1 else "помірний перекос"}',
              fontweight='bold')
ax4.set_xlabel('Сума депозиту, €'); ax4.set_ylabel('Щільність')
ax4.legend(fontsize=8.5, ncol=2, framealpha=0.9)
ax4.set_facecolor('#ffffff')
ax4.spines['top'].set_visible(False); ax4.spines['right'].set_visible(False)

# 5. Sample size
ax5 = fig.add_subplot(gs[1, 2])
mde_range = np.linspace(5, 45, 300)
ns = [sample_size(conv_b, m) for m in mde_range]
ax5.plot(mde_range, ns, color='#533FB7', linewidth=2.2)
ax5.axhline(min(n_a, n_b), color='tomato', linestyle='--', linewidth=1.8,
            label=f'Поточний n={min(n_a, n_b):,}')
ax5.axvline(lift_conv, color=COLOR_A, linestyle=':', linewidth=1.8,
            label=f'Ліфт {lift_conv:.1f}%')
ax5.fill_between(mde_range, ns, min(n_a, n_b),
                 where=[n <= min(n_a, n_b) for n in ns],
                 alpha=0.12, color='green', label='Вибірки вистачає')
ax5.set_title('Sample size vs MDE\n(power=80%, α=0.05)', fontweight='bold')
ax5.set_xlabel('MDE, % ліфту'); ax5.set_ylabel('n на групу')
ax5.legend(fontsize=8.5)
ax5.set_yscale('log')
ax5.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax5.set_facecolor('#ffffff')
ax5.spines['top'].set_visible(False); ax5.spines['right'].set_visible(False)

fig.suptitle('A/B Тест: Бонусний лист → Перший депозит', fontsize=14, fontweight='bold')
plt.savefig('ab_test_results.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('📊 Графік збережено: ab_test_results.png')

## 12. Фінальний підсумок

In [ ]:
def verdict(p):
    if p < 0.05:  return '✅ Значущий'
    if p < 0.10:  return '⚠️  На межі'
    return '❌ НЕ значущий'

summary = pd.DataFrame([
    {'Тест': 'Z-тест (конверсія)',         'Статистика': f'Z={z_stat:.3f}',    'p-value': f'{p_z_two:.4f}',   'Результат': verdict(p_z_two)},
    {'Тест': 'T-тест Велча (серед. деп)',  'Статистика': f't={t_welch:.3f}',   'p-value': f'{p_welch:.4f}',   'Результат': verdict(p_welch)},
    {'Тест': 'T-тест Стьюдента (серед.)',  'Статистика': f't={t_student:.3f}', 'p-value': f'{p_student:.4f}', 'Результат': verdict(p_student)},
    {'Тест': 'Mann-Whitney (медіана)',      'Статистика': f'U={u_stat:.0f}',    'p-value': f'{p_mw:.4f}',      'Результат': verdict(p_mw)},
])
display(summary)

print(f"""
─────────────────────────────────────────────────────
  ВИСНОВОК
─────────────────────────────────────────────────────
  Ліфт конверсії:        {lift_conv:+.2f}%  (A: {conv_a:.3%} vs B: {conv_b:.3%})
  Ліфт середнього деп:   {lift_mean:+.2f}%  (A: €{a_dep.mean():.2f} vs B: €{b_dep.mean():.2f})
  Поточна потужність:    {actual_power:.1%}
  Потрібно для впевненості: {n_needed:,} на групу (є {min(n_a,n_b):,})
  Рекомендований t-тест: {chosen_test} (Levene p={p_levene:.4f})
─────────────────────────────────────────────────────
""")